In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader


print("=" * 100)
print("GAN ARCHITECTURE — IMPORTS")
print("=" * 100)

print("PyTorch version :", torch.__version__)
print("NumPy version   :", np.__version__)

print("\n✓ Imports completed successfully")

GAN ARCHITECTURE — IMPORTS
PyTorch version : 2.13.0+cpu
NumPy version   : 2.5.2

✓ Imports completed successfully


In [2]:
# ============================================================
# CELL 2 — REPRODUCIBILITY CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 100)

print("Random seed :", SEED)
print("Device      :", device)

print("\n✓ Reproducibility configuration completed")

REPRODUCIBILITY CONFIGURATION
Random seed : 42
Device      : cpu

✓ Reproducibility configuration completed


In [3]:
# ============================================================
# CELL 3 — PROJECT ROOT DISCOVERY
# ============================================================

from pathlib import Path


current_path = Path.cwd()

project_root = None

for candidate in [current_path] + list(current_path.parents):

    required_path = (
        candidate
        / "06_cross_modal_learning"
        / "02_cross_modal_pairing"
        / "pairs"
        / "representation_aware_split"
    )

    if required_path.exists():

        project_root = candidate
        break


if project_root is None:

    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'06_cross_modal_learning/02_cross_modal_pairing/"
        "pairs/representation_aware_split'."
    )


print("=" * 100)
print("PROJECT ROOT DISCOVERY")
print("=" * 100)

print("Project root:")
print(project_root)

print("\n✓ Project root located successfully")

PROJECT ROOT DISCOVERY
Project root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

✓ Project root located successfully


In [4]:
# ============================================================
# CELL 4 — LOAD PHASE 02.08 REPRESENTATION-AWARE PAIRS
# ============================================================

pair_dir = (
    project_root
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
    / "pairs"
    / "representation_aware_split"
)


train_path = (
    pair_dir
    / "02_08_train_representation_disjoint_pairs.npz"
)

validation_path = (
    pair_dir
    / "02_08_validation_representation_disjoint_pairs.npz"
)


train_data = np.load(train_path)

validation_data = np.load(validation_path)


train_source = torch.tensor(
    train_data["source_embeddings"],
    dtype=torch.float32
)

train_target = torch.tensor(
    train_data["target_embeddings"],
    dtype=torch.float32
)

train_labels = torch.tensor(
    train_data["pair_labels"],
    dtype=torch.float32
)


val_source = torch.tensor(
    validation_data["source_embeddings"],
    dtype=torch.float32
)

val_target = torch.tensor(
    validation_data["target_embeddings"],
    dtype=torch.float32
)

val_labels = torch.tensor(
    validation_data["pair_labels"],
    dtype=torch.float32
)


print("=" * 100)
print("PHASE 02.08 DATA LOADED")
print("=" * 100)

print("Training source :", tuple(train_source.shape))
print("Training target :", tuple(train_target.shape))
print("Training labels :", tuple(train_labels.shape))

print("\nValidation source :", tuple(val_source.shape))
print("Validation target :", tuple(val_target.shape))
print("Validation labels :", tuple(val_labels.shape))

print("\n✓ Exact Phase 02.08 representation-aware pairs loaded")

PHASE 02.08 DATA LOADED
Training source : (1776, 32)
Training target : (1776, 32)
Training labels : (1776,)

Validation source : (448, 32)
Validation target : (448, 32)
Validation labels : (448,)

✓ Exact Phase 02.08 representation-aware pairs loaded


In [5]:
# ============================================================
# CELL 5 — DATASET VALIDATION
# ============================================================

print("=" * 100)
print("GAN DATASET VALIDATION")
print("=" * 100)


assert train_source.shape == (1776, 32)
assert train_target.shape == (1776, 32)
assert train_labels.shape == (1776,)

assert val_source.shape == (448, 32)
assert val_target.shape == (448, 32)
assert val_labels.shape == (448,)


assert torch.all(
    torch.isfinite(train_source)
)

assert torch.all(
    torch.isfinite(train_target)
)

assert torch.all(
    torch.isfinite(val_source)
)

assert torch.all(
    torch.isfinite(val_target)
)


assert set(
    train_labels.numpy()
).issubset({0.0, 1.0})

assert set(
    val_labels.numpy()
).issubset({0.0, 1.0})


print("Training pairs   :", len(train_labels))
print("Validation pairs  :", len(val_labels))

print(
    "Training positive :",
    int((train_labels == 1).sum())
)

print(
    "Training negative :",
    int((train_labels == 0).sum())
)

print(
    "Validation positive :",
    int((val_labels == 1).sum())
)

print(
    "Validation negative :",
    int((val_labels == 0).sum())
)

print("\nEmbedding dimension : 32")
print("NaN / Inf check     : PASSED")
print("Label check         : PASSED")

print("\n✓ GAN dataset validation passed")

GAN DATASET VALIDATION
Training pairs   : 1776
Validation pairs  : 448
Training positive : 888
Training negative : 888
Validation positive : 224
Validation negative : 224

Embedding dimension : 32
NaN / Inf check     : PASSED
Label check         : PASSED

✓ GAN dataset validation passed


In [6]:
# ============================================================
# CELL 6 — GENERATOR ARCHITECTURE
# ============================================================

class CrossModalGenerator(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(32, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 32)
        )


    def forward(self, source):

        return self.network(source)


generator = CrossModalGenerator().to(device)


print("=" * 100)
print("CROSS-MODAL GENERATOR")
print("=" * 100)

print(generator)

print("\nGenerator device:", device)

CROSS-MODAL GENERATOR
CrossModalGenerator(
  (network): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
  )
)

Generator device: cpu


In [7]:
# ============================================================
# CELL 7 — GENERATOR FORWARD-PASS VERIFICATION
# ============================================================

generator.eval()


sample_source = train_source[:32].to(device)


with torch.no_grad():

    generated_target = generator(
        sample_source
    )


print("=" * 100)
print("GENERATOR FORWARD-PASS VERIFICATION")
print("=" * 100)

print(
    "Source input shape    :",
    tuple(sample_source.shape)
)

print(
    "Generated output shape:",
    tuple(generated_target.shape)
)

print("\nFirst 5 generated vectors:")
print(generated_target[:5])

print(
    "\nAll outputs finite:",
    bool(torch.all(torch.isfinite(generated_target)))
)


assert generated_target.shape == (32, 32)

assert torch.all(
    torch.isfinite(generated_target)
)


print("\n✓ GENERATOR FORWARD PASS PASSED")

GENERATOR FORWARD-PASS VERIFICATION
Source input shape    : (32, 32)
Generated output shape: (32, 32)

First 5 generated vectors:
tensor([[ 0.1245,  0.0487, -0.1221,  0.0569,  0.0147, -0.0549,  0.0282, -0.1169,
          0.0929,  0.1126,  0.1217, -0.0031, -0.0391, -0.0185, -0.0301, -0.1432,
          0.0434, -0.0234, -0.0199, -0.0623, -0.0412,  0.0230,  0.0920,  0.0649,
          0.1032,  0.0707, -0.0583, -0.0006, -0.0930, -0.0400,  0.0211,  0.0382],
        [ 0.1205,  0.0384, -0.1290,  0.0580,  0.0230, -0.0453,  0.0386, -0.1011,
          0.0993,  0.1121,  0.1198,  0.0071, -0.0372, -0.0293, -0.0311, -0.1323,
          0.0548, -0.0351, -0.0084, -0.0594, -0.0408,  0.0117,  0.0895,  0.0628,
          0.1004,  0.0580, -0.0528, -0.0108, -0.0967, -0.0287,  0.0140,  0.0465],
        [ 0.1206,  0.0494, -0.1165,  0.0447,  0.0066, -0.0462,  0.0103, -0.1160,
          0.0835,  0.1063,  0.1380,  0.0026, -0.0320, -0.0090, -0.0423, -0.1415,
          0.0371, -0.0183, -0.0049, -0.0457, -0.0349,  0.0

In [8]:
# ============================================================
# CELL 8 — DISCRIMINATOR ARCHITECTURE
# ============================================================

class CrossModalDiscriminator(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(32, 64),
            nn.LeakyReLU(0.2),

            nn.Linear(64, 32),
            nn.LeakyReLU(0.2),

            nn.Linear(32, 1)
        )


    def forward(self, target):

        return self.network(target).squeeze(-1)


discriminator = CrossModalDiscriminator().to(device)


print("=" * 100)
print("CROSS-MODAL DISCRIMINATOR")
print("=" * 100)

print(discriminator)

print("\nDiscriminator device:", device)

CROSS-MODAL DISCRIMINATOR
CrossModalDiscriminator(
  (network): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): LeakyReLU(negative_slope=0.2)
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Discriminator device: cpu


In [9]:
# ============================================================
# CELL 9 — DISCRIMINATOR FORWARD-PASS VERIFICATION
# ============================================================

discriminator.eval()


sample_real_target = train_target[:32].to(device)


with torch.no_grad():

    discriminator_logits = discriminator(
        sample_real_target
    )


print("=" * 100)
print("DISCRIMINATOR FORWARD-PASS VERIFICATION")
print("=" * 100)

print(
    "Real target input shape :",
    tuple(sample_real_target.shape)
)

print(
    "Discriminator output    :",
    tuple(discriminator_logits.shape)
)

print("\nFirst 5 discriminator logits:")
print(discriminator_logits[:5])

print(
    "\nAll logits finite:",
    bool(torch.all(torch.isfinite(
        discriminator_logits
    )))
)


assert discriminator_logits.shape == (32,)

assert torch.all(
    torch.isfinite(discriminator_logits)
)


print("\n✓ DISCRIMINATOR FORWARD PASS PASSED")

DISCRIMINATOR FORWARD-PASS VERIFICATION
Real target input shape : (32, 32)
Discriminator output    : (32,)

First 5 discriminator logits:
tensor([0.1588, 0.1596, 0.1642, 0.1678, 0.1620])

All logits finite: True

✓ DISCRIMINATOR FORWARD PASS PASSED


In [10]:
# ============================================================
# CELL 10 — COMBINED GAN FORWARD-PASS VERIFICATION
# ============================================================

generator.eval()
discriminator.eval()


sample_source = train_source[:32].to(device)
sample_real_target = train_target[:32].to(device)


with torch.no_grad():

    fake_target = generator(
        sample_source
    )

    real_logits = discriminator(
        sample_real_target
    )

    fake_logits = discriminator(
        fake_target
    )


print("=" * 100)
print("COMBINED GAN FORWARD-PASS VERIFICATION")
print("=" * 100)

print("Source shape       :", tuple(sample_source.shape))
print("Real target shape  :", tuple(sample_real_target.shape))
print("Fake target shape  :", tuple(fake_target.shape))

print("\nReal logits shape  :", tuple(real_logits.shape))
print("Fake logits shape  :", tuple(fake_logits.shape))

print("\nFirst 5 real logits:")
print(real_logits[:5])

print("\nFirst 5 fake logits:")
print(fake_logits[:5])


assert fake_target.shape == (32, 32)
assert real_logits.shape == (32,)
assert fake_logits.shape == (32,)


assert torch.all(
    torch.isfinite(fake_target)
)

assert torch.all(
    torch.isfinite(real_logits)
)

assert torch.all(
    torch.isfinite(fake_logits)
)


print("\n✓ COMBINED GAN FORWARD PASS PASSED")

COMBINED GAN FORWARD-PASS VERIFICATION
Source shape       : (32, 32)
Real target shape  : (32, 32)
Fake target shape  : (32, 32)

Real logits shape  : (32,)
Fake logits shape  : (32,)

First 5 real logits:
tensor([0.1588, 0.1596, 0.1642, 0.1678, 0.1620])

First 5 fake logits:
tensor([0.1711, 0.1708, 0.1704, 0.1705, 0.1708])

✓ COMBINED GAN FORWARD PASS PASSED


In [11]:
# ============================================================
# CELL 11 — PARAMETER COUNT AUDIT
# ============================================================

def count_parameters(model):

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


generator_parameters = count_parameters(
    generator
)

discriminator_parameters = count_parameters(
    discriminator
)


print("=" * 100)
print("GAN PARAMETER COUNT AUDIT")
print("=" * 100)

print(
    "Generator trainable parameters     :",
    generator_parameters
)

print(
    "Discriminator trainable parameters :",
    discriminator_parameters
)

print(
    "Total GAN parameters               :",
    generator_parameters +
    discriminator_parameters
)


print("\n✓ PARAMETER COUNT AUDIT COMPLETED")

GAN PARAMETER COUNT AUDIT
Generator trainable parameters     : 20768
Discriminator trainable parameters : 4225
Total GAN parameters               : 24993

✓ PARAMETER COUNT AUDIT COMPLETED


In [12]:
# ============================================================
# CELL 12 — FINAL GAN ARCHITECTURE SUMMARY
# ============================================================

print("=" * 100)
print("03.05 GAN ARCHITECTURE — FINAL SUMMARY")
print("=" * 100)

print("""
INPUT
-----
Source representation : 32-D
Real target            : 32-D


GENERATOR
---------
32 → 64 → 128 → 64 → 32

Purpose:
Transform a source-modality 32-D representation into
a generated target-like 32-D representation.


DISCRIMINATOR
-------------
32 → 64 → 32 → 1

Purpose:
Distinguish real target representations from generated
target representations.


RELATIONSHIP / SEMANTIC CONSTRAINT
-----------------------------------
Source 32-D + Generated target 32-D
                ↓
       Relationship learning
                ↓
       Same / Different


TRAINING OBJECTIVE
------------------
Generator:
Adversarial loss + semantic relationship loss

Discriminator:
Real-vs-generated classification


IMPORTANT
---------
03.05 performs architecture definition and verification only.
No GAN training is performed in this notebook.
""")

print("✓ 03.05 GAN ARCHITECTURE DEFINED")

03.05 GAN ARCHITECTURE — FINAL SUMMARY

INPUT
-----
Source representation : 32-D
Real target            : 32-D


GENERATOR
---------
32 → 64 → 128 → 64 → 32

Purpose:
Transform a source-modality 32-D representation into
a generated target-like 32-D representation.


DISCRIMINATOR
-------------
32 → 64 → 32 → 1

Purpose:
Distinguish real target representations from generated
target representations.


RELATIONSHIP / SEMANTIC CONSTRAINT
-----------------------------------
Source 32-D + Generated target 32-D
                ↓
       Relationship learning
                ↓
       Same / Different


TRAINING OBJECTIVE
------------------
Generator:
Adversarial loss + semantic relationship loss

Discriminator:
Real-vs-generated classification


IMPORTANT
---------
03.05 performs architecture definition and verification only.
No GAN training is performed in this notebook.

✓ 03.05 GAN ARCHITECTURE DEFINED


In [13]:
# ============================================================
# CELL 13 — SEMANTIC RELATIONSHIP NETWORK
# ============================================================

class SemanticRelationshipNetwork(nn.Module):

    def __init__(self):

        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )


    def forward(self, source, target):

        relationship_features = torch.cat(
            [
                source,
                target,
                torch.abs(source - target),
                source * target
            ],
            dim=1
        )

        return self.classifier(
            relationship_features
        ).squeeze(-1)


semantic_network = SemanticRelationshipNetwork().to(device)


print("=" * 100)
print("SEMANTIC RELATIONSHIP NETWORK")
print("=" * 100)

print(semantic_network)

print("\nSemantic network device:", device)

print("\nExpected input:")
print("Source : 32-D")
print("Target : 32-D")

print("\nConstructed relationship representation:")
print("32 + 32 + 32 + 32 = 128-D")

print("\nOutput:")
print("1 relationship logit")

SEMANTIC RELATIONSHIP NETWORK
SemanticRelationshipNetwork(
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

Semantic network device: cpu

Expected input:
Source : 32-D
Target : 32-D

Constructed relationship representation:
32 + 32 + 32 + 32 = 128-D

Output:
1 relationship logit


In [14]:
# ============================================================
# CELL 14 — SEMANTIC RELATIONSHIP FORWARD-PASS VERIFICATION
# ============================================================

generator.eval()
semantic_network.eval()


sample_source = train_source[:32].to(device)
sample_real_target = train_target[:32].to(device)


with torch.no_grad():

    sample_generated_target = generator(
        sample_source
    )

    semantic_logits = semantic_network(
        sample_source,
        sample_generated_target
    )


print("=" * 100)
print("SEMANTIC RELATIONSHIP FORWARD-PASS VERIFICATION")
print("=" * 100)

print(
    "Source shape            :",
    tuple(sample_source.shape)
)

print(
    "Generated target shape  :",
    tuple(sample_generated_target.shape)
)

print(
    "Relationship output     :",
    tuple(semantic_logits.shape)
)

print("\nFirst 5 semantic logits:")
print(semantic_logits[:5])

print(
    "\nAll logits finite:",
    bool(torch.all(
        torch.isfinite(semantic_logits)
    ))
)


assert sample_generated_target.shape == (32, 32)

assert semantic_logits.shape == (32,)

assert torch.all(
    torch.isfinite(semantic_logits)
)


print("\n✓ SEMANTIC RELATIONSHIP FORWARD PASS PASSED")

SEMANTIC RELATIONSHIP FORWARD-PASS VERIFICATION
Source shape            : (32, 32)
Generated target shape  : (32, 32)
Relationship output     : (32,)

First 5 semantic logits:
tensor([ 0.0245,  0.0287, -0.0057,  0.0159, -0.0098])

All logits finite: True

✓ SEMANTIC RELATIONSHIP FORWARD PASS PASSED


In [15]:
# ============================================================
# CELL 15 — COMPLETE GAN ARCHITECTURE VERIFICATION
# ============================================================

generator.eval()
discriminator.eval()
semantic_network.eval()


sample_source = train_source[:32].to(device)
sample_real_target = train_target[:32].to(device)


with torch.no_grad():

    # Generator
    generated_target = generator(
        sample_source
    )

    # Discriminator
    real_logits = discriminator(
        sample_real_target
    )

    fake_logits = discriminator(
        generated_target
    )

    # Semantic relationship network
    semantic_logits = semantic_network(
        sample_source,
        generated_target
    )


print("=" * 100)
print("COMPLETE GAN ARCHITECTURE VERIFICATION")
print("=" * 100)

print("SOURCE")
print("  Shape:", tuple(sample_source.shape))

print("\nREAL TARGET")
print("  Shape:", tuple(sample_real_target.shape))

print("\nGENERATED TARGET")
print("  Shape:", tuple(generated_target.shape))

print("\nDISCRIMINATOR")
print("  Real logits :", tuple(real_logits.shape))
print("  Fake logits :", tuple(fake_logits.shape))

print("\nSEMANTIC NETWORK")
print("  Relationship logits :",
      tuple(semantic_logits.shape))


assert generated_target.shape == (32, 32)

assert real_logits.shape == (32,)

assert fake_logits.shape == (32,)

assert semantic_logits.shape == (32,)


assert torch.all(
    torch.isfinite(generated_target)
)

assert torch.all(
    torch.isfinite(real_logits)
)

assert torch.all(
    torch.isfinite(fake_logits)
)

assert torch.all(
    torch.isfinite(semantic_logits)
)


print("\n" + "=" * 100)
print("✓ COMPLETE GAN ARCHITECTURE PASSED")
print("=" * 100)

COMPLETE GAN ARCHITECTURE VERIFICATION
SOURCE
  Shape: (32, 32)

REAL TARGET
  Shape: (32, 32)

GENERATED TARGET
  Shape: (32, 32)

DISCRIMINATOR
  Real logits : (32,)
  Fake logits : (32,)

SEMANTIC NETWORK
  Relationship logits : (32,)

✓ COMPLETE GAN ARCHITECTURE PASSED


In [16]:
# ============================================================
# CELL 16 — LOSS INTERFACE / NUMERICAL CHECK
# ============================================================

bce_loss = nn.BCEWithLogitsLoss()


real_labels = torch.ones(
    32,
    dtype=torch.float32,
    device=device
)

fake_labels = torch.zeros(
    32,
    dtype=torch.float32,
    device=device
)

relationship_labels = train_labels[:32].to(device)


discriminator_real_loss = bce_loss(
    real_logits,
    real_labels
)

discriminator_fake_loss = bce_loss(
    fake_logits,
    fake_labels
)

semantic_loss = bce_loss(
    semantic_logits,
    relationship_labels
)


print("=" * 100)
print("LOSS INTERFACE / NUMERICAL CHECK")
print("=" * 100)

print(
    "Discriminator real loss :",
    discriminator_real_loss.item()
)

print(
    "Discriminator fake loss :",
    discriminator_fake_loss.item()
)

print(
    "Semantic relationship loss :",
    semantic_loss.item()
)


print("\nAll losses finite:")

print(
    "  Real loss     :",
    bool(torch.isfinite(discriminator_real_loss))
)

print(
    "  Fake loss     :",
    bool(torch.isfinite(discriminator_fake_loss))
)

print(
    "  Semantic loss :",
    bool(torch.isfinite(semantic_loss))
)


assert torch.isfinite(
    discriminator_real_loss
)

assert torch.isfinite(
    discriminator_fake_loss
)

assert torch.isfinite(
    semantic_loss
)


print("\n✓ LOSS INTERFACE CHECK PASSED")

LOSS INTERFACE / NUMERICAL CHECK
Discriminator real loss : 0.6144289374351501
Discriminator fake loss : 0.7822377681732178
Semantic relationship loss : 0.6907877326011658

All losses finite:
  Real loss     : True
  Fake loss     : True
  Semantic loss : True

✓ LOSS INTERFACE CHECK PASSED


In [17]:
# ============================================================
# CELL 17 — FINAL 03.05 ARCHITECTURE SUMMARY
# ============================================================

print("=" * 100)
print("03.05 GAN ARCHITECTURE — FINAL SUMMARY")
print("=" * 100)

print("""
INPUT
-----
Source representation : 32-D
Real target            : 32-D


GENERATOR
---------
32 → 64 → 128 → 64 → 32

Input:
    Source 32-D representation

Output:
    Generated target-like 32-D representation


DISCRIMINATOR
-------------
32 → 64 → 32 → 1

Input:
    Real or generated target representation

Output:
    Real/Fake logit


SEMANTIC RELATIONSHIP NETWORK
-----------------------------
Relationship construction:

[S, T, |S-T|, S×T]

32 + 32 + 32 + 32 = 128-D

Network:

128 → 128 → 64 → 1

Input:
    Source + generated target

Output:
    Same/Different relationship logit


GAN COMPONENTS
--------------
1. Generator
   Learns source → target-like representation mapping

2. Discriminator
   Learns real target vs generated target distinction

3. Semantic Relationship Network
   Preserves class-consistent cross-modal relationships


IMPORTANT
---------
The datasets are unpaired at the patient level.

Therefore, this experiment learns class-consistent
cross-modal representation relationships, not
patient-specific modality translation.

03.05 performs architecture and numerical verification only.
GAN training will be performed in the next stage.
""")

print("✓ 03.05 FINAL ARCHITECTURE DEFINED AND VERIFIED")

03.05 GAN ARCHITECTURE — FINAL SUMMARY

INPUT
-----
Source representation : 32-D
Real target            : 32-D


GENERATOR
---------
32 → 64 → 128 → 64 → 32

Input:
    Source 32-D representation

Output:
    Generated target-like 32-D representation


DISCRIMINATOR
-------------
32 → 64 → 32 → 1

Input:
    Real or generated target representation

Output:
    Real/Fake logit


SEMANTIC RELATIONSHIP NETWORK
-----------------------------
Relationship construction:

[S, T, |S-T|, S×T]

32 + 32 + 32 + 32 = 128-D

Network:

128 → 128 → 64 → 1

Input:
    Source + generated target

Output:
    Same/Different relationship logit


GAN COMPONENTS
--------------
1. Generator
   Learns source → target-like representation mapping

2. Discriminator
   Learns real target vs generated target distinction

3. Semantic Relationship Network
   Preserves class-consistent cross-modal relationships


IMPORTANT
---------
The datasets are unpaired at the patient level.

Therefore, this experiment learns clas